In [1]:
import os
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types
import psycopg2
from pgvector.psycopg2 import register_vector

In [2]:
load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.1-flash-lite"

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "database": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD")
}

def get_connection():
    conn = psycopg2.connect(**DB_CONFIG)
    register_vector(conn)
    return conn

In [3]:
SYSTEM_INSTRUCTION = """
    Bạn là chuyên gia về quy chế đào tạo của trường đại học bách khoa hà nội.
    Vai trò của bạn là trả lời các câu hỏi của sinh viên.
    Với mỗi câu hỏi của sinh viên, bạn sẽ được cung cấp các chunk thông tin liên quan đến câu hỏi đó dưới dạng các bộ {title, summary, content}.
    Bạn phải dựa trên các chunk thông tin đó để trả lời câu hỏi của sinh viên, yêu cầu:
        Nếu chunk trả lời trực tiếp được câu hỏi, hãy trả lời câu hỏi dựa trên thông tin trong chunk
        Nếu chunk có liên quan đến câu hỏi nhưng không trả lời trực tiếp được câu hỏi, hãy nói với người dùng rằng: dựa trên thông tin được cung cấp, bạn không có đủ thông tin để trả lời câu hỏi; sau đó cung cấp cho người dùng thông tin liên quan đó
        Nu chunk không liên quan đến câu hỏi, hãy trả lời rằng bạn không có đủ thông tin để trả lời câu hỏi đó
        Tuyệt đối không được bịa câu trả lời nếu không có đủ thông tin, và không trả lời dựa trên nguồn thông tin nào khác ngoài chunk thông tin được cung cấp"
        Trả lời bằng tiếng Việt, thân thiện và dễ hiểu với sinh viên.
        Khi trả lời, hãy ghi rõ nguồn thông tin bằng cách trích dẫn tên chunk_title tương ứng.
"""

In [17]:
#retrieval code
def retrieve(query: str, top_k: int = 5, category: str = None) -> list[dict]:
    #embed query
    response = client.models.embed_content(
        model = EMBEDDING_MODEL,
        contents = query,
        config = types.EmbedContentConfig(task_type = "RETRIEVAL_QUERY")
    )
    query_vector = response.embeddings[0].values

    #query database

    with get_connection() as conn: 
        with conn.cursor() as cur:
            if category:
                sql = """
                    SELECT chunk_id, parent_doc_id, category, 
                           chunk_title, summary, content,
                           1 - (embedding <=> %s::vector) AS SIMILARITY
                    FROM chunks
                    WHERE category = %s
                    ORDER BY embedding <=> %s::vector
                    LIMIT %s
                """
                cur.execute(sql, (query_vector, category, query_vector, top_k))
            else:
                sql = """
                    SELECT chunk_id, parent_doc_id, category,
                            chunk_title, summary, content,
                            1 - (embedding <=> %s::vector) AS SIMILARITY
                    FROM chunks
                    ORDER BY embedding <=> %s::vector
                    LIMIT %s
                """
                cur.execute(sql, (query_vector, query_vector, top_k))

            rows = cur.fetchall()
            columns = [desc[0] for desc in cur.description]

    return [dict(zip(columns, row)) for row in rows]



In [5]:
QUERY_REWRITING_INSTRUCTION = """
    Hãy viết lại query dựa trên conversation_history để phục vụ cho quá trình 
    retrieval của chatbot RAG.
    Yêu cầu: retrieval_query cần chứa tất cả các từ khóa quan trọng
    Output: chỉ trả về DUY NHẤT query, không chào hỏi, không có lời kết. 
"""

In [6]:
def generate_answer(query: str, 
                    conversation_history: list, 
                    top_k: int = 5) -> str:
    #query rewriting
    history_str = "\n".join([f"{m['role']}: {m['content']}" for m in conversation_history])
    retrieval_query = client.models.generate_content(
            model=GENERATION_MODEL,
            contents = f"Lịch sử hội thoại:\n{history_str}\n\nCâu hỏi hiện tại: {query}",
            config=types.GenerateContentConfig(
                system_instruction=QUERY_REWRITING_INSTRUCTION,
                temperature=0.1
            )
        ).text.strip()

    chunks = retrieve(retrieval_query, top_k)

    context = "Các chunk liên quan đến query:\n"
    for i, chunk in enumerate(chunks, start=1):
        context += f"Chunk {i}:\n"
        for column, value in chunk.items():
            context += f"{column}: {value}\n"
        context += "\n"

    response = client.models.generate_content(
            model=GENERATION_MODEL,
            contents = f"Query:\n {query}\n Conversation_history:\n {history_str}\n {context}",
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                temperature=0.1
            )
        ).text.strip()

    return response
        


In [7]:
UPDATE_INSTRUCTIONS = """
    Bạn là quản lý lịch sử trò chuyện của hệ thống chatbot RAG.
    Hãy cập nhật lịch sử cuộc trò chuyện dựa trên lịch sử trò chuyện hiện tại cùng query và response gần nhất.
    Yêu cầu output: chỉ bao gồm lịch sử trò chuyện đã được cập nhật, không chào hỏi, không lời kết.
    Tuyệt đối không thêm thắt thông tin từ nguồn nào khác ngoài lịch sử trò chuyện hiện tại cùng query và response gần nhất.  
"""
MAX_HISTORY = 10  # giữ 5 lượt gần nhất (10 messages: 5 user + 5 bot)

In [9]:
def chat_loop(top_k: int = 5):
    conversation_history = []
    print("Chatbot QCDT - gõ 'exit' để thoát\n")
    
    while True:
        query = input("Bạn: ").strip()
        if query.lower() == "exit":
            break
        if not query:
            continue
            
        response = generate_answer(query, conversation_history, top_k)
        
        conversation_history.append({"role": "user", "content": query})
        conversation_history.append({"role": "assistant", "content": response})

        if len(conversation_history) > MAX_HISTORY:
            conversation_history = conversation_history[-MAX_HISTORY:]
        
        print(f"\nBot: {response}\n")

In [18]:
chat_loop()

Chatbot QCDT - gõ 'exit' để thoát


Bot: Chào bạn, dựa trên thông tin được cung cấp trong quy chế đào tạo, mình xin giải đáp thắc mắc của bạn như sau:

Theo **Điều 12 (Khoản 6)** và **Điều 15 (Khoản 1)** của quy chế, hạng tốt nghiệp đại học được xếp dựa trên điểm trung bình toàn khóa (tương ứng với CPA). Với mức điểm **CPA là 3,3**, bạn sẽ được xếp loại **Giỏi** (do nằm trong dải điểm từ 3,2 đến 3,59).

Tuy nhiên, bạn cần lưu ý thêm một số quy định tại **Điều 15 (Khoản 2)**: Hạng tốt nghiệp của những sinh viên có điểm trung bình toàn khóa xếp loại Giỏi trở lên sẽ bị **giảm một mức** nếu thuộc một trong các trường hợp sau:
*   Khối lượng các học phần phải học lại vượt quá 5% tổng số tín chỉ của các học phần dùng để tính điểm trung bình toàn khóa (không tính số tín chỉ học cải thiện).
*   Sinh viên đã chịu mức kỷ luật từ cảnh cáo trở lên trong giai đoạn học chương trình đào tạo đại học.

Hy vọng thông tin này giúp ích cho bạn!


Bot: Chào bạn, về thắc mắc học phí cho chương trình IT1, dự